# 04 分类训练：基线与最终方案

Edited by Wang Hongjie

XGBoost
- A：6 features
- B：11 features, including 6 original features and 5 cross-market features

Classification task: Determine whether the rate of return exceeds 0.2%.
$$
y=
\begin{cases}
1,&r>0.002\\
0,&r\leq0.002
\end{cases}
$$

## 0. Function Definition and Basic Configuration

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_curve
import hashlib
import json
import logging
import os
import platform
import shutil
import sys
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo
import joblib
import sklearn
from sklearn.metrics import accuracy_score, average_precision_score, balanced_accuracy_score, confusion_matrix, f1_score, r2_score, roc_auc_score
from IPython.display import display

# Run from SP500; all notebook helpers are local.
ROOT = Path.cwd().resolve()
if not (ROOT / "01_data_preprocessing.ipynb").is_file():
    raise FileNotFoundError("Run this notebook from the SP500 directory.")

### Path Config

In [ ]:
DATA_DIR = ROOT / "data"

MARKET_DIR = DATA_DIR / "market"

CLEAN_FILE = DATA_DIR / "processed" / "sp500_clean.csv"

FEATURE_DIR = DATA_DIR / "features"

RAW_FILE = DATA_DIR / "raw" / "sp500_daily_1996-11-13_to_2026-11-12_raw.json"

ORIGINAL = [
    "return_1d", "return_5d", "return_20d", "volatility_20d",
    "intraday_return", "intraday_range",
]

MARKET = [
    "vix_close", "vix_change", "treasury_2y_change_bp",
    "dxy_change_pct", "wti_change_usd",
]

BP = 10_000     # basis points - unit. 1 BP = 0.01% = 0.0001

MARKET_FILES = [
    MARKET_DIR / "vix_history.csv",
    *[MARKET_DIR / name for name in (
        "treasury_archive.csv", "treasury_2025.csv", "treasury_example.csv",
        "dxy.json", "wti.json",
    )],
]

### Table, image and file saving

In [ ]:
def display_path(path):
    """Show relative project paths and external filenames."""
    path = Path(path)
    if not path.is_absolute():
        return path.as_posix()
    try:
        return Path(os.path.relpath(path, ROOT)).as_posix()
    except ValueError:
        return path.name

def save_json(path, value):
    """Write UTF-8 JSON without nonstandard NaN or Infinity."""
    def convert(item):
        if isinstance(item, np.generic):
            return item.item()
        if isinstance(item, (Path, pd.Timestamp, datetime)):
            return str(item)
        raise TypeError(f"Cannot serialize {type(item)}")
    Path(path).write_text(
        json.dumps(value, ensure_ascii=False, indent=2, default=convert, allow_nan=False),
        encoding="utf-8",
    )

def save_csv(path, frame, index=True):
    frame.to_csv(path, index=index, encoding="utf-8-sig", index_label="signal_date" if index else None)

def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def show_table(frame, decimals=4):
    """Display rounded values without changing saved data."""
    display(frame.round(decimals))

### 读取上游数据与构造标签

读取已经准备的数据；标签与跨市场特征按原时间约定构造。


In [ ]:
def load_prepared_samples(task):
    """Load step 02 samples and verify upstream inputs."""
    if task not in ("classification", "regression"):
        raise ValueError("Unknown task.")
    path = FEATURE_DIR / f"{task}_samples.csv"
    manifest_file = FEATURE_DIR / "feature_build.json"
    if not path.is_file() or not manifest_file.is_file():
        raise FileNotFoundError("Run steps 01 and 02 first.")
    manifest = json.loads(manifest_file.read_text(encoding="utf-8"))
    for source in manifest["sources"]:
        if file_hash(ROOT / source["path"]) != source["sha256"]:
            raise ValueError("Upstream inputs changed; rerun step 02.")
    if file_hash(path) != manifest["outputs"][path.name]:
        raise ValueError("Feature file changed; rerun step 02.")
    samples = pd.read_csv(path, index_col="signal_date",
                          parse_dates=["signal_date", "trade_date", "settle_date"],
                          float_precision="round_trip")
    samples["target_class"] = samples["target_class"].astype("Int8")
    assert samples.index.is_monotonic_increasing and samples.index.is_unique
    return samples


### 按时间划分并剔除越界标签

沿用有效样本的 8:1:1 边界，剔除跨分区收益标签。


In [ ]:
def chronological_split(samples, ratios=(0.8, 0.1, 0.1)):
    """Split valid samples by date; drop targets crossing boundaries."""
    if len(ratios) != 3 or not np.isclose(sum(ratios), 1) or min(ratios) <= 0:
        raise ValueError("Provide three positive ratios summing to one.")
    val_size, test_size = (round(len(samples) * x) for x in ratios[1:])
    train_size = len(samples) - val_size - test_size
    if min(train_size, val_size, test_size) < 2:
        raise ValueError("Too few samples in a split.")
    validation_start = samples.iloc[train_size]["trade_date"]
    test_start = samples.iloc[train_size + val_size]["trade_date"]
    splits = {
        "train": samples.loc[samples.settle_date < validation_start].copy(),
        "val": samples.loc[(samples.trade_date >= validation_start) & (samples.settle_date < test_start)].copy(),
        "test": samples.loc[samples.trade_date >= test_start].copy(),
    }
    assert all(len(frame) > 0 for frame in splits.values())
    assert splits["train"].settle_date.max() <= splits["val"].index.min()
    assert splits["val"].settle_date.max() <= splits["test"].index.min()
    all_dates = pd.Index(np.concatenate([f.index.to_numpy() for f in splits.values()]))
    assert all_dates.is_unique, "Duplicate signal dates across splits."
    summary = pd.DataFrame([
        {"split": name, "samples": len(frame), "first_signal": frame.index.min(),
         "last_signal": frame.index.max(), "first_entry": frame.trade_date.min(),
         "last_settlement": frame.settle_date.max()}
        for name, frame in splits.items()
    ])
    return splits, summary


### 实验目录与逐轮训练日志

每次运行新建目录，保存配置、数据指纹、代码快照和逐轮日志。


In [ ]:
@dataclass
class Run:
    path: Path
    config: dict
    logger: logging.Logger

def start_run(name, config, sources):
    """Create a new run directory without overwriting results."""
    stamp = datetime.now(ZoneInfo("Asia/Singapore")).strftime("%Y%m%d_%H%M%S_%f")
    path = ROOT / "output" / name / stamp
    path.mkdir(parents=True, exist_ok=False)
    logger = logging.getLogger(f"{name}.{stamp}")
    logger.setLevel(logging.INFO)
    logger.propagate = False
    # Keep the notebook default warning format.
    fmt = logging.Formatter("%(asctime)s | %(message)s", datefmt="%H:%M:%S")
    for handler in (logging.FileHandler(path / "run.log", encoding="utf-8"), logging.StreamHandler(sys.stdout)):
        handler.setFormatter(fmt)
        logger.addHandler(handler)
    config = dict(config, scenario=name, created_at=datetime.now(ZoneInfo("Asia/Singapore")).isoformat())
    save_json(path / "config.json", config)
    save_json(path / "environment.json", {
        "python": platform.python_version(), "python_executable": sys.executable,
        "numpy": np.__version__, "pandas": pd.__version__, "scikit_learn": sklearn.__version__,
        "xgboost": xgb.__version__, "timezone": "Asia/Singapore",
    })
    save_json(path / "data_sources.json", [
        {"path": str(p), "sha256": file_hash(p), "bytes": p.stat().st_size}
        for p in sources
    ])
    snapshot = path / "code"
    snapshot.mkdir()
    for source in sorted(ROOT.glob("0[1-7]_*.ipynb")):
        if source.exists():
            shutil.copy2(source, snapshot / source.name)
    logger.info("Started %s; output directory %s", name, display_path(path))
    return Run(path, config, logger)

def stage_log(run, directory, message):
    directory.mkdir(parents=True, exist_ok=True)
    message = str(message)
    run.logger.info(message)
    with (directory / "stage.log").open("a", encoding="utf-8") as stream:
        stream.write(f"{datetime.now().isoformat(timespec='seconds')} | {message}\n")

class TrainingLog(xgb.callback.TrainingCallback):
    """Write each XGBoost round to the display and run logs."""
    def __init__(self, run, directory, every=1):
        self.run, self.directory, self.every = run, directory, every

    def after_iteration(self, model, epoch, evals_log):
        if epoch == 0 or (epoch + 1) % self.every == 0:
            values = [f"{part}.{metric}={scores[-1]:.8f}"
                      for part, metrics in evals_log.items() for metric, scores in metrics.items()]
            stage_log(self.run, self.directory, f"Round {epoch + 1} | " + " | ".join(values))
        return False

class LoggedEarlyStopping(xgb.callback.EarlyStopping):
    """Log metrics before checking early stopping, including the final round."""
    def __init__(self, run, directory, log_every=1, **kwargs):
        super().__init__(**kwargs)
        self.progress = TrainingLog(run, directory, every=log_every)

    def after_iteration(self, model, epoch, evals_log):
        self.progress.after_iteration(model, epoch, evals_log)
        return super().after_iteration(model, epoch, evals_log)


### 模型保存、重新加载与指标计算

保存训练和验证结果，测试时重新加载模型，并按原定义计算指标。


In [ ]:
def predict_model(model, frame, columns, unit):
    if isinstance(model, xgb.Booster):
        result = model.predict(xgb.DMatrix(frame[columns], nthread=4))
    else:
        result = model.predict(frame[columns])
    return result / BP if unit == "basis_points" else result

def measure(task, actual, prediction, threshold=None):
    if task == "classification":
        decisions = prediction >= threshold
        return {
            "roc_auc": roc_auc_score(actual, prediction),
            "ap": average_precision_score(actual, prediction),
            "accuracy": accuracy_score(actual, decisions),
            "balanced_accuracy": balanced_accuracy_score(actual, decisions),
            "f1": f1_score(actual, decisions, zero_division=0),
        }
    error = (prediction - actual) * BP
    return {"mae_bp": np.abs(error).mean(), "rmse_bp": np.sqrt(np.mean(error ** 2)), "r2": r2_score(actual, prediction)}

def new_training_dir(run, period):
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    parent = run.path if period == "holdout" else run.path / "folds" / period
    folder = parent / f"train_{stamp}"
    folder.mkdir(parents=True, exist_ok=False)
    return folder

def save_training(run, period, folder, model, splits, columns, selected, history=None):
    """Save model and train/validation predictions before testing."""
    folder.mkdir(parents=True, exist_ok=True)
    task = run.config["task"]
    selected = dict(selected, task=task, features=list(columns),
                    train_mean=float(splits["train"].target_return.mean()),
                    train_median=float(splits["train"].target_return.median()))
    save_json(folder / "selected.json", selected)
    if isinstance(model, xgb.Booster):
        model.save_model(folder / "model.ubj")
    else:
        joblib.dump(model, folder / "model.joblib")
    if history:
        save_json(folder / "history.json", history)
        history_frame = pd.DataFrame({f"{part}_{metric}": values
                                      for part, metrics in history.items() for metric, values in metrics.items()})
        history_frame.index += 1
        history_frame.index.name = "round"
        history_frame.to_csv(folder / "history.csv", encoding="utf-8-sig")
    label = "target_class" if task == "classification" else "target_return"
    metrics = {}
    for split in ("train", "val"):
        frame = splits[split]
        prediction = predict_model(model, frame, columns, selected["prediction_unit"])
        output = frame[["trade_date", "settle_date", label]].assign(prediction=prediction)
        save_csv(folder / f"{split}_predictions.csv", output)
        metrics[split] = measure(task, frame[label].to_numpy(dtype=float), prediction, selected.get("threshold"))
    save_json(folder / "metrics.json", metrics)
    stage_log(run, folder, f"{period} training complete | selected {selected.get('alpha', selected.get('best_rounds'))}")
    return {"folder": folder, "selected": selected}

def evaluate_saved(run, period, artifact, test):
    """Reload the model and create a fresh test directory."""
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    parent = run.path if period == "holdout" else run.path / "folds" / period
    folder = parent / f"test_{stamp}"
    folder.mkdir(parents=True, exist_ok=False)
    selected = artifact["selected"]
    source = artifact["folder"]
    if (source / "model.ubj").exists():
        model = xgb.Booster(params={"nthread": 4})
        model.load_model(source / "model.ubj")
    else:
        model = joblib.load(source / "model.joblib")
    columns, task = selected["features"], selected["task"]
    stage_log(run, folder, f"{period} testing started | samples {len(test)} | model loaded from disk")
    prediction = predict_model(model, test, columns, selected["prediction_unit"])
    if not np.isfinite(prediction).all():
        raise ValueError("Model produced invalid predictions.")
    label = "target_class" if task == "classification" else "target_return"
    actual = test[label].to_numpy(dtype=float)
    metrics = {"period": period, "n_test": len(test), **measure(task, actual, prediction, selected.get("threshold"))}
    predictions = test[["trade_date", "settle_date", label]].assign(prediction=prediction)
    if task == "classification":
        predictions["predicted_class"] = (prediction >= selected["threshold"]).astype(int)
        metrics.update(threshold=selected["threshold"], positive_rate=float(actual.mean()))
        save_csv(folder / "confusion_matrix.csv", pd.DataFrame(
            confusion_matrix(actual, predictions.predicted_class, labels=[0, 1]),
            columns=["predicted_0", "predicted_1"]), index=False)
    else:
        # Use train median for MAE and train mean for RMSE baselines.
        predictions["constant_mean"] = selected["train_mean"]
        predictions["constant_median"] = selected["train_median"]
        metrics["constant_mae_bp"] = np.abs(actual - selected["train_median"]).mean() * BP
        metrics["constant_rmse_bp"] = np.sqrt(np.mean((actual - selected["train_mean"]) ** 2)) * BP
        metrics["delta_mae_bp"] = metrics["mae_bp"] - metrics["constant_mae_bp"]
        metrics["delta_rmse_bp"] = metrics["rmse_bp"] - metrics["constant_rmse_bp"]
    save_csv(folder / "predictions.csv", predictions)
    save_json(folder / "metrics.json", metrics)
    save_json(folder / "model_source.json", {"training_directory": str(source), "selected": selected})
    stage_log(run, folder, f"{period} testing complete | " + " | ".join(
        f"{key}={metrics[key]:.4f}" for key in (("roc_auc", "ap") if task == "classification" else ("mae_bp", "rmse_bp"))))
    return metrics, predictions


## 1. 导入与配置


In [2]:
# Reuse tree parameters; tune rounds and the classification threshold.
# Keep the original search range.

XGB_PARAMS = {
    "tree_method": "hist",       # Histogram tree growth for daily data
    "max_depth": 1,              # Limit tree complexity
    "eta": 0.03,                 # Per-round learning rate
    "min_child_weight": 20,    # Original notebook value (legacy script used 5)
    "subsample": 0.8,            # Sample training rows each round
    "colsample_bytree": 0.9,     # Sample features for each tree
    "lambda": 5.0,               # L2 regularization
    "seed": 42, "nthread": 4,
}
MAX_ROUNDS = 400
EARLY_STOPPING = 30              # Stop after 30 validation rounds without improvement

SCENARIO = "classification_comparison"
ARM_LABELS = {'baseline': 'Arm A · 6 features + XGBoost', 'final': 'Arm B · 11 features + XGBoost'}
FEATURE_SETS = {"baseline": ORIGINAL, "final": ORIGINAL + MARKET}
CONFIG = {
    "task": "classification", "evaluation": "single_chronological_holdout",
    "split_ratios": [0.8, 0.1, 0.1], "horizon_trading_days": 1,
    "positive_threshold": 0.002, "label": "open[t+2] / open[t+1] - 1",
    "last_data_date": "2026-11-12", "seed": 42,
    "tree_params": XGB_PARAMS, "max_rounds": MAX_ROUNDS, "early_stopping": EARLY_STOPPING,
}
RUN = start_run(SCENARIO, CONFIG, [RAW_FILE, CLEAN_FILE, FEATURE_DIR / "classification_samples.csv", FEATURE_DIR / "feature_build.json"] + MARKET_FILES)
arm_runs = {}
for arm in ARM_LABELS:
    folder = RUN.path / arm
    folder.mkdir()
    arm_config = dict(CONFIG, arm=ARM_LABELS[arm], features=FEATURE_SETS[arm])
    save_json(folder / "config.json", arm_config)
    arm_runs[arm] = Run(folder, arm_config, RUN.logger)


22:06:37 | Started classification_comparison; output directory output/classification_comparison/20261123_220637_250154


## 2. 特征与标签

两组使用相同日期。A 有 6 项价格特征，B 增加 5 项跨市场特征；标签为 `open[t+2]/open[t+1]-1 > 0.002`。


In [4]:
samples = load_prepared_samples("classification")  # Verify upstream file hashes before training
save_csv(RUN.path / "features_and_targets.csv", samples)
show_table(samples[ORIGINAL + ["target_return", "target_class"]].tail(3))


,return_1d,return_5d,return_20d,volatility_20d,intraday_return,intraday_range,target_return,target_class
signal_date,,,,,,,,
2026-11-06,0.0031,-0.0268,-0.0556,0.0125,-0.0001,0.0101,0.0122,1
2026-11-09,0.0235,0.0011,-0.0330,0.0138,0.0172,0.0209,0.0098,1
2026-11-10,-0.0038,-0.0063,-0.0365,0.0138,0.0010,0.0098,-0.0047,0


## 3. 单次按时间划分
按时间顺序将数据划分为训练集 : 验证集 : 测试集 -- 80% : 10% : 10%。两种方案使用相同的数据划分。


In [ ]:
# Pass configured ratios to the chronological splitter.
splits, split_summary = chronological_split(samples, CONFIG["split_ratios"])  # Remove labels crossing time boundaries
save_csv(RUN.path / "split_summary.csv", split_summary, index=False)
show_table(split_summary)
RUN.logger.info("Chronological split: train %s / validation %s / test %s",
                *(len(splits[k]) for k in ("train", "val", "test")))


,split,samples,first_signal,last_signal,first_entry,last_settlement
0,train,3593,2009-01-02,2023-04-12,2009-01-05,2023-04-14
1,val,448,2023-04-14,2025-01-27,2023-04-17,2025-01-29
2,test,449,2025-01-29,2026-11-10,2025-01-30,2026-11-12


22:06:37 | Chronological split: train 3593 / validation 448 / test 449


## 4. 定义 XGBoost 训练与调参函数
A/B 两方案 共用 **XGBoost** 函数，仅特征不同。验证 AUC 选最佳轮次，验证平衡准确率选分类阈值。函数只接收训练和验证数据。


In [6]:
def train_xgboost(run, stage, train, val, columns):
    """Tune on training and validation data only."""
    folder = new_training_dir(run, stage)
    stage_log(run, folder, f"{stage} XGBoost training started | train={len(train)} / val={len(val)}")
    label = "target_class"
    dtrain = xgb.DMatrix(train[columns], label=train[label], nthread=4)
    dval = xgb.DMatrix(val[columns], label=val[label], nthread=4)
    params = dict(XGB_PARAMS, objective="binary:logistic", eval_metric="auc")
    history = {}

    # Use a fresh callback; keep trees through the best validation round.
    model = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=MAX_ROUNDS,
        evals=[(dtrain, "train"), (dval, "val")],
        evals_result=history,
        callbacks=[
            # Log each round before the early-stop check.
            LoggedEarlyStopping(run, folder, log_every=1,
                rounds=EARLY_STOPPING, metric_name="auc",
                data_name="val", maximize=True, save_best=True,
            ),
        ],
        verbose_eval=False,
    )
    selected = {
        "params": params, "best_rounds": int(model.best_iteration + 1),
        "trained_rounds": len(history["val"]["auc"]),
        "validation_metric": "auc", "validation_score": float(model.best_score),
        "prediction_unit": "probability",
        "training_window": "Same fixed training set",
    }

    # Select threshold on validation only; maximizing TPR-FPR maximizes balanced accuracy.
    val_probability = model.predict(dval)
    fpr, tpr, thresholds = roc_curve(val[label], val_probability)
    finite = np.isfinite(thresholds)
    threshold = float(thresholds[finite][np.argmax((tpr - fpr)[finite])])  # Choose threshold on validation data only
    selected["threshold"] = threshold
    stage_log(run, folder, f"Best validation round {selected['best_rounds']}; classification threshold {threshold:.6f}")

    stage_log(run, folder, f"Trained {selected['trained_rounds']} rounds; retained best {selected['best_rounds']}-round model")
    return save_training(run, stage, folder, model, {"train": train, "val": val},
                         columns, selected, history)


## 训练
分别训练两种方案，并根据验证结果调整模型。设置 *`early_stopping`*：如果连续 30 轮没有改善，则提前结束训练。最后保存模型和训练日志。


In [7]:
artifacts = {}  # Train each arm independently
for arm in ARM_LABELS:
    RUN.logger.info("Training arm: %s", ARM_LABELS[arm])
    artifacts[arm] = train_xgboost(
        arm_runs[arm], "holdout", splits["train"], splits["val"], FEATURE_SETS[arm])


22:06:37 | Training arm: Arm A · 6 features + XGBoost
22:06:37 | holdout XGBoost training started | train=3593 / val=448
22:06:37 | Round 1 | train.auc=0.55038873 | val.auc=0.50133939
22:06:37 | Round 2 | train.auc=0.57030861 | val.auc=0.51446952
22:06:37 | Round 3 | train.auc=0.57046783 | val.auc=0.51540304
22:06:37 | Round 4 | train.auc=0.57752680 | val.auc=0.51202411
22:06:37 | Round 5 | train.auc=0.58114453 | val.auc=0.51357659
22:06:37 | Round 6 | train.auc=0.58091724 | val.auc=0.51377953
22:06:37 | Round 7 | train.auc=0.58062570 | val.auc=0.51481451
22:06:37 | Round 8 | train.auc=0.58040221 | val.auc=0.51501745
22:06:37 | Round 9 | train.auc=0.58277952 | val.auc=0.51547406
22:06:37 | Round 10 | train.auc=0.58257471 | val.auc=0.51555524
22:06:37 | Round 11 | train.auc=0.58237275 | val.auc=0.51600170
22:06:37 | Round 12 | train.auc=0.58242847 | val.auc=0.51640758
22:06:37 | Round 13 | train.auc=0.58558339 | val.auc=0.51544362
22:06:37 | Round 14 | train.auc=0.58558118 | val.auc=0.5

## 测试
读取已保存模型，在同一个测试集上分别评价。测试数据不参与轮次、分类阈值或 alpha 的选择。


In [8]:
rows = []
for arm in ARM_LABELS:
    RUN.logger.info("Testing arm: %s", ARM_LABELS[arm])
    metrics, predictions = evaluate_saved(arm_runs[arm], "holdout", artifacts[arm], splits["test"])
    metrics.pop("period", None)
    rows.append(dict(scenario=ARM_LABELS[arm], **metrics))
    save_csv(arm_runs[arm].path / "test_predictions.csv", predictions)
    save_json(arm_runs[arm].path / "summary.json", metrics)
comparison = pd.DataFrame(rows)  # Compare the same held-out dates
save_csv(RUN.path / "comparison_summary.csv", comparison, index=False)
save_json(RUN.path / "status.json", {"status": "completed", "evaluation": "single_chronological_holdout"})


22:06:39 | Testing arm: Arm A · 6 features + XGBoost
22:06:39 | holdout testing started | samples 449 | model loaded from disk
22:06:39 | holdout testing complete | roc_auc=0.5426 | ap=0.4631
22:06:39 | Testing arm: Arm B · 11 features + XGBoost
22:06:39 | holdout testing started | samples 449 | model loaded from disk
22:06:39 | holdout testing complete | roc_auc=0.5576 | ap=0.4602


## 同一测试集的结果对比实验
`ROC-AUC`、`AP` 数值越高越好。


In [ ]:
show_table(comparison[["scenario", "n_test", "roc_auc", "ap", "balanced_accuracy", "f1"]])
print("Output directory:", display_path(RUN.path))

,scenario,n_test,roc_auc,ap,balanced_accuracy,f1
0,Arm A · 6 features + XGBoost,449,0.5426,0.4631,0.5307,0.4866
1,Arm B · 11 features + XGBoost,449,0.5576,0.4602,0.5682,0.5523


Output directory: output/classification_comparison/20261123_220637_250154


## 文献对照

[Qin (2026)](https://doi.org/10.70267/aitia.2026412420) 使用不同数据和标签，指标仅作背景参考。


In [1]:
SECTION='04_classification'
import json, hashlib
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().resolve()
assert (ROOT/'01_data_preprocessing.ipynb').is_file()
LIT_OUT = ROOT/'output/literature'/SECTION/datetime.now().strftime('%Y%m%d_%H%M%S_%f')
LIT_OUT.mkdir(parents=True)
runs=sorted(p for p in (ROOT/'output/classification_comparison').iterdir() if (p/'status.json').exists() and json.loads((p/'status.json').read_text())['status']=='completed')
run=runs[-1];rows=[];INPUTS=[]
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, balanced_accuracy_score, f1_score
for name,arm in [('Local Class A','baseline'),('Local Class B','final')]:
    path=run/arm/'test_predictions.csv';INPUTS.append(path)
    f=pd.read_csv(path);y=f.target_class;p=f.prediction;d=f.predicted_class
    rows.append({'origin':'local','model':name,'n':len(f),'first_signal':f.signal_date.min(),'last_signal':f.signal_date.max(),'ROC_AUC':roc_auc_score(y,p),'accuracy':accuracy_score(y,d),'F1':f1_score(y,d,zero_division=0),'AP':average_precision_score(y,p),'balanced_accuracy':balanced_accuracy_score(y,d),'test_majority_accuracy':max(y.mean(),1-y.mean()),'label':'open-to-open >0.002'})
rows += [{'origin':'paper reported','model':'Qin XGBoost','n':None,'first_signal':'NR','last_signal':'NR','ROC_AUC':.5393,'accuracy':.5360,'F1':.5550,'AP':None,'balanced_accuracy':None,'test_majority_accuracy':.5690,'label':'next adjusted close > current adjusted close'}]
PAPER={'authors':'Mingyu Qin','year':2026,'doi':'10.70267/aitia.2026412420','url':'https://journals.zeuspress.org/index.php/conference/article/download/1343/1306','location':'Table 1, PDF p5 / printed p416','publication':'conference paper','unit':'fractions, not percentages','not_reported':['AP','balanced accuracy','exact test dates','test n'],'not_reproduced':True}
literature_table=pd.DataFrame(rows)

literature_table.to_csv(LIT_OUT/'literature_comparison.csv',index=False,encoding='utf-8-sig')
(LIT_OUT/'extraction.json').write_text(json.dumps({'checked_on':'2026-11-24','paper':PAPER,'table':literature_table.replace({np.nan:None}).to_dict('records'),'input_sha256':{str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in INPUTS}},ensure_ascii=False,indent=2),encoding='utf-8')
display(literature_table.fillna('N/A'))
print('Saved:',LIT_OUT.relative_to(ROOT))


,origin,model,n,first_signal,last_signal,ROC_AUC,accuracy,F1,AP,balanced_accuracy,test_majority_accuracy,label
0,local,Local Class A,449.0,2025-01-29,2026-11-10,0.542628,0.530067,0.486618,0.46305,0.530738,0.583519,open-to-open >0.002
1,local,Local Class B,449.0,2025-01-29,2026-11-10,0.557589,0.552339,0.552339,0.460206,0.568192,0.583519,open-to-open >0.002
2,paper reported,Qin XGBoost,N/A,NR,NR,0.539300,0.536000,0.555000,N/A,N/A,0.569000,next adjusted close > current adjusted close


Saved: output\literature\04_classification\20261124_155356_402767
